# Notebook 2 — Math Refresher: Fundamental Functions in Neuroscience

*Quantitative Methods in Neuroscience, Master in Neuroscience, University of Geneva (2026-27)*

This is the practical companion to **Lecture 2 — Calculus Refresher & Fundamental Functions for Neuroscience**. The lecture introduced a small zoo of mathematical objects that show up over and over in experimental neuroscience: exponentials, logarithms, Gaussians, sigmoids, sines and cosines, and the calculus tools (derivatives and integrals) used to manipulate them.

In this notebook we will **see some of those functions at work on a concrete neuroscience dataset**, and along the way we will pick up some of the basic Python building blocks needed for the rest of the course.

### By the end of this notebook you will be able to:
- Load a behavioural dataset from a `.csv` file and inspect it with `pandas`.
- Write your own Python **functions** and **loops**, and operate on data with **NumPy arrays** and **Pandas DataFrames**.
- Plot data and overlay mathematical functions with **Matplotlib**.
- Use **sigmoid**, **exponential**, **logarithm** and **Gaussian** functions to describe perceptual decision data.
- Compute **numerical derivatives** and **integrals** to summarize *rate of change* and *accumulation* on real data.
- Connect each math step to a concrete neuroscience interpretation.

### Table of contents
1. [The scientific context: a 2AFC perceptual decision task](#section-1)
2. [Setting up the workspace and loading the data](#section-2)
3. [Python building blocks: functions, loops, arrays, plots](#section-3)
   - 3.5 [Reusing your code: putting functions in a module](#section-3-5)
4. [Fundamental functions in action](#section-4)
   - 4.1 [The sigmoid: psychometric curves](#section-4-1)
   - 4.2 [Exponential and logarithm: reaction time distributions](#section-4-2)
   - 4.3 [Gaussian: the bell-shaped friend](#section-4-3)
   - 4.4 [Sine and cosine: a quick LFP-flavoured detour](#section-4-4)
5. [Learning curves and a bit of calculus](#section-5)
   - 5.1 [Accuracy across sessions: an exponential learning curve](#section-5-1)
   - 5.2 [Numerical derivative: rate of learning](#section-5-2)
   - 5.3 [Numerical integral: cumulative correct responses](#section-5-3)
6. [Lego-style exercises](#section-6)
7. [About this notebook](#about)

> **How to read this notebook.** Markdown cells like this one explain *why* we are doing something and the *concept* behind the math. Code cells implement the concept. Look out for **`▶ Task`** boxes: those are short exercises where you have to edit or extend the code. If you are stuck ask the TAs. If you decide to use AI help treat it as a tutor, not as a replacement for thinking: you should still be able to read each line of code we produce and explain what it does (and why). In this case we suggest to use the custom GPT bot we prepared for this course: https://chatgpt.com/g/g-69fc9bfc9a748191a912a124800d9254-qmn-teaching-assistant-bot


<a id="section-1"></a>
## 1. The scientific context: a 2AFC perceptual decision task

Many cognitive and systems neuroscience experiments use a deceptively simple paradigm: a participant (human, monkey, or rodent) is shown an ambiguous stimulus and asked to pick between **two alternatives**. For instance, *"is this cloud of moving dots drifting to the left or to the right?"*. This is called a **two-alternative forced choice** (2AFC) task.

By varying the **strength** of the stimulus (here, the *coherence* of the moving dots, i.e. what fraction of dots actually move in a consistent direction), the experimenter can probe how perception scales with evidence. Two classical quantities come out of such an experiment:
- **Accuracy**: the fraction of correct responses, plotted as a function of stimulus strength. This is the **psychometric curve**, and it typically has an S-shape (a **sigmoid**).
- **Reaction time (RT)**: the time it takes the subject to commit to an answer. RTs are typically *positively skewed*, longer on harder trials, and well approximated by a **log-normal** distribution.

Across **training sessions**, well-practiced participants get faster and more accurate: their psychometric curve becomes steeper and their RTs shorter. The trajectory of accuracy across sessions is called a **learning curve**, and is often well described by an **exponential approach** to an asymptote.

> *Why this dataset?* TO BE REPLACED WITH A REAL DATASET.

### The dataset

The file `data/psychophysics_2afc.csv` contains the trial-by-trial behaviour of **3 simulated subjects** (`S01`, `S02`, `S03`) tested on **8 sessions** each. Each row is one trial with the following columns:

| Column             | Meaning                                                                  |
|--------------------|--------------------------------------------------------------------------|
| `trial_id`         | Unique trial index in the whole file.                                    |
| `subject_id`       | Identifier of the subject (`S01`, `S02`, `S03`).                          |
| `session`          | Session number (1 to 8).                                                  |
| `trial_in_session` | Position of the trial within the session.                                |
| `coherence`        | Signed motion coherence in [-1, 1]. Negative = leftward, positive = rightward, magnitude = strength of the motion signal. |
| `response`         | Subject's choice: `0` = "left", `1` = "right".                            |
| `correct`          | `1` if the choice matched the sign of the coherence, `0` otherwise; `NaN` at coherence 0 (no objectively correct answer). |
| `rt_s`             | Reaction time, in seconds.                                                |


<a id="section-2"></a>
## 2. Setting up the workspace and loading the data

> **Before you run anything**, make sure you have activated the course conda environment **`qmn`** in your terminal (or selected it as the kernel of this notebook in VS Code). The environment is described in `environment.yml` at the root of the course folder; the full step-by-step conda walkthrough is in Notebook 1 (Week 1 — Setup & Python intro). If you have not done it yet, the short version is:
>
> ```bash
> conda env create -f ../environment.yml
> conda activate qmn
> python -m ipykernel install --user --name qmn --display-name "Python (qmn)"
> ```

Once the kernel is ready, we need to **import** the Python libraries we'll rely on:
- `numpy`: fast numerical arrays and math functions (`np.exp`, `np.log`, `np.sin`, …).
- `pandas`: tables of mixed-type data, with column labels (a bit like a smart Excel sheet).
- `matplotlib.pyplot`: plotting.

The convention is to import them under short aliases (`np`, `pd`, `plt`): you'll see this in every Python project you read.


In [ ]:
# --- Import the core scientific Python stack
import numpy as np # for numerical computing
import pandas as pd # for data manipulation and analysis
import matplotlib.pyplot as plt # for plotting
from cycler import cycler # for custom color in plots

# A little bit of style for the plots (feel free to tweak later!)
plt.rcParams.update({
    "figure.figsize": (6, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "axes.prop_cycle": cycler(color=[
        "#1f77b4",  # blue
        "#ff7f0e",  # orange
        "#5b2c83",  # purple
    ]),
})

print(f"NumPy      : {np.__version__}")
print(f"pandas     : {pd.__version__}")
print(f"matplotlib : {plt.matplotlib.__version__}")


Now we load the data with `pandas.read_csv`. `pandas` returns a **DataFrame**.

A **DataFrame** is like a spreadsheet inside Python: it has rows and columns, and each column has a name. In neuroscience data, one row might represent one trial, one neuron, one time point, or one animal, depending on the dataset. Each column then stores a variable, such as firing rate, stimulus condition, reaction time, calcium signal, or animal ID.

This is different from a NumPy **array**, which is usually just a block of numbers arranged in one or more dimensions. Arrays are excellent for mathematical operations, linear algebra, filtering, and simulations. But they do not naturally store column names or mixed data types.

A DataFrame is useful when the data are **structured**: for example, when we want to keep together numerical variables, labels, conditions, and metadata.

We can inspect it using:

```python
df.head()       # show the first rows
df.shape        # number of rows and columns
df.dtypes       # data type of each column
df.describe()   # summary statistics for numerical columns
```

A useful way to think about it is:

- `NumPy array`  = good for numerical computation https://numpy.org/doc/stable/reference/generated/numpy.array.html
- `DataFrame`    = good for labelled, table-like data https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html

Later, if we only need the numerical values for computation, we can convert selected DataFrame columns into a NumPy array.

In [ ]:
# --- Load the synthetic 2AFC dataset
df = pd.read_csv("data/psychophysics_2afc.csv")

print("Shape (rows, columns):", df.shape)
df.head(10)

In [ ]:
# --- A first look at the columns and their basic statistics
print(df.dtypes)
print()
df.describe()

> **▶ Task 2.1 — Inspect the dataset**
>
> Using the cell below:
>
> 1. Print how many trials each subject contributed.  
>    Hint: `df["subject_id"].value_counts()`
>
> 2. Print the **list of unique coherence values** used in the experiment.  
>    Hint: `np.unique(...)`
>
> 3. Print the avaerage accuracy of a subject.  
>    Hint: `.mean()`
>

> **Useful syntax tips**
>
> - `.items()` on a pandas Series or dictionary returns an iterable of tuples:
>
>   ```python
>   (index_or_key, value)
>   ```
>
>   Useful when looping with `for`.
>
> - `np.sort(...)` reorders a NumPy array.
>
>   Useful for displaying values in increasing order.
>
> - `df[df["subject_id"] == "S01"]` selects only the rows where a condition is true.
>
>   Useful for filtering the data from one subject.
>
> - `df["correct"].mean()` computes the mean of a pandas DataFrame column.
>
>   Useful for computing accuracy when the column contains `0` for incorrect trials and `1` for correct trials.

In [ ]:
# TODO 2.1 — your code here
# 1. trials per subject
unique_subjects = df["subject_id"].value_counts() # TO BE REMOVED
for subject_id, count in unique_subjects.items(): # TO BE REMOVED
    print(f"Subject {subject_id} has {count} trials.") # TO BE REMOVED

# 2. unique coherences (ordered from lowest to highest)
unique_coherence_levels = df["coherence"].unique() # TO BE REMOVED
print("Unique coherence levels:", np.sort(unique_coherence_levels)) # TO BE REMOVED

# 3. overall accuracy of subject 1
subject_1_data = df[df["subject_id"] == "S01"] # TO BE REMOVED
accuracy_subject_1 = subject_1_data["correct"].mean() # TO BE REMOVED
print(f"Overall accuracy of subject 1: {accuracy_subject_1:.2%}") # TO BE REMOVED

<a id="section-3"></a>
## 3. Python building blocks: functions, loops, arrays, plots

The rest of the notebook will keep coming back to four ideas. We collect them here so we can reuse them comfortably.

### 3.1 Functions

A **function** is a reusable block of code. You define it with `def`, give it some inputs (*parameters*), and tell it what to return. Think of a function as a small machine: feed it numbers in, get numbers out.

Functions are the most powerful tool we have for *not repeating ourselves*. If you find yourself copy-pasting code, that's a hint that you should turn it into a function. This makes you save time and allows you to rapidly chenge your logic by modifying the code in a single place (instead of may scattered snippets of code!) in case of need.


In [ ]:
# --- A very first function
def square(x):
    """Return x squared."""
    return x * x

print(square(3))     # 9
print(square(0.5))   # 0.25

Many of the *mathematical functions* from the lecture have a direct Python translation. Let's write one of them right now, the **logistic sigmoid**, because we'll need it in Section 4.

Mathematically:

$$\sigma(x; \, k, x_0) \;=\; \frac{1}{1 + e^{-k\,(x - x_0)}}$$

where $k$ controls how *steep* the S-curve is and $x_0$ shifts it horizontally.

In [ ]:
# --- A logistic sigmoid
def sigmoid(x, slope=1.0, bias=0.0):
    """Logistic sigmoid with adjustable slope and horizontal bias."""
    return 1.0 / (1.0 + np.exp(-slope * (x - bias)))

# Try it on a single number
print(sigmoid(0.0))      # 0.5 — by construction
print(sigmoid(10.0))     # ~1.0
print(sigmoid(-10.0))    # ~0.0

### 3.2 For loops

A **`for` loop** repeats the same operation on each item of a sequence. It is the right tool when you want to do the same thing for every subject, every condition, every trial.

In [ ]:
# --- Loop over the three subjects and print their mean accuracy
for subj in df["subject_id"].unique():
    mean_acc = df.loc[df["subject_id"] == subj, "correct"].mean()
    print(f"{subj}: mean accuracy = {mean_acc:.3f}")

### 3.3 NumPy arrays and vectorization

A **NumPy array** is a grid of numbers, all of the same type. Unlike a Python list, you can do math on a whole array at once: no loop required. This is called **vectorization**, and it is both faster *and* more readable. Doing operations with arrays in loops (one at a time) teh amout of time required to perform them would grow with the number of elements we have to manipulate, potentially making our code very slow!

A function written with NumPy operations (`np.exp`, `+`, `-`, `*`, …) is automatically **vectorized**: calling it on an array applies it element-wise. Our `sigmoid` above already has this property.

In [ ]:
# --- Evaluate sigmoid on a whole grid of x values in one shot
x_grid = np.linspace(-1.0, 1.0, 201)     # 201 evenly spaced points from -1 to 1
y_grid = sigmoid(x_grid, slope=8.0, bias=0.0)

print("x_grid shape :", x_grid.shape)
print("y_grid shape :", y_grid.shape)
print("First 5 values of x_grid:", x_grid[:5])
print("First 5 values of y_grid:", y_grid[:5].round(3))

### 3.4 Plotting

`matplotlib` works like a paint-by-numbers kit: you create a figure, add elements (lines, dots, labels), then `show` it. Below we plot the sigmoid we just built.

In [ ]:
# --- Plot the sigmoid
fig, ax = plt.subplots()
ax.plot(x_grid, y_grid, lw=2)
ax.axhline(0.5, color="gray", lw=0.8, ls="--")
ax.axvline(0.0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("input  x")
ax.set_ylabel(r"$\sigma(x)$")
ax.set_title("A logistic sigmoid (slope=8, bias=0)")
plt.show()


> **▶ Task 3.1 — Play with the building blocks.**
> 1. Write a function `my_gaussian(x, mu, sigma)` that returns the value of an un-normalized Gaussian
>    $$ g(x) \;=\; \exp\!\left(-\tfrac{(x-\mu)^2}{2\sigma^2}\right). $$
>    Use only NumPy operations: your function should work on a single number *or* on an array, with no extra effort.
> 2. Evaluate it on `x_grid` for `mu = 0.2`, `sigma = 0.1` and plot the result.
> 3. Using a `for` loop, plot the Gaussian for three different `sigma` values (`0.05`, `0.1`, `0.3`) on the *same* axes. Add a legend.


In [ ]:
# TODO 3.1 — your code here

def my_gaussian(x, mu, sigma):
    # TODO: return the un-normalized Gaussian evaluated at x
    gauss = np.exp(-0.5 * ((x - mu) / sigma) ** 2) # TO BE REMOVED
    return gauss

# Plot one Gaussian for mu=0.2, sigma=0.1 on x_grid (this should show a bell-shaped curve)
fig, ax = plt.subplots()
ax.plot(x_grid, my_gaussian(x_grid, mu=0.2, sigma=0.1), lw=2, color="#3a12ec", label="mu=0.2, sigma=0.1")
ax.legend()
ax.set_xlabel("x"); ax.set_ylabel("g(x)")
ax.set_title("Un-normalized Gaussian")
plt.show()

# TODO: plot three Gaussians (sigma = 0.05, 0.1, 0.3) on the same axes using a for loop
sigmas = [0.05, 0.1, 0.3] # TO BE REMOVED
mu = 0.2 # TO BE REMOVED
fig, ax = plt.subplots() # TO BE REMOVED
for sigma in sigmas: # TO BE REMOVED
    ax.plot(x_grid, my_gaussian(x_grid, mu=mu, sigma=sigma), lw=2, label=f"mu={mu}, sigma={sigma}") # TO BE REMOVED
ax.legend() # TO BE REMOVED
ax.set_xlabel("x"); ax.set_ylabel("g(x)") # TO BE REMOVED
ax.set_title("Un-normalized Gaussians with different sigmas") # TO BE REMOVED
plt.show() # TO BE REMOVED

<a id="section-3-5"></a>
### 3.5 Reusing your code: putting functions in a module

Right now `sigmoid` lives *inside* this notebook. That is fine for one-off exploration, but the moment we want to reuse the function in *another* notebook (next week, in Notebook 3) we would have to copy-paste the definition or hunt the right cell to re-run. Copy-paste is the easiest way to introduce subtle bugs.

The standard practice in every real Python project is to put reusable helper functions in a separate `.py` file (a "**module**") and `import` them, just like we do with NumPy or pandas. We will follow this convention throughout the course.

We have created an `src/` folder next to this notebook and dropped the `sigmoid` function into `src/qmn_utils.py`. The cell below prints the file's contents. You can also open it in VS Code (`notebooks/src/qmn_utils.py`) to see it side-by-side with this notebook.

**Note**: Reusing the same function across different parts of the code, instead of redefining it multiple times, is an essential "code hygiene" habit. It keeps the codebase more readable, easier to verify and modify as the project grows in size.


In [ ]:
# --- The contents of src/qmn_utils.py
from pathlib import Path # for reading the file contents at a specified path
print(Path("src/qmn_utils.py").read_text())

Now we can import functions from this module exactly like we import from NumPy or pandas. The dot syntax `src.qmn_utils` follows the folder structure: `src/` is a *package*, `qmn_utils` is a *module* inside it, and `sigmoid` is a *name* inside the module.


In [ ]:
# --- Import sigmoid from our own module and check it agrees with the inline version
from src.qmn_utils import sigmoid as sigmoid_from_module

print(sigmoid(0.0),  sigmoid_from_module(0.0))   # both 0.5
print(sigmoid(2.0),  sigmoid_from_module(2.0))   # both ~0.88
print(sigmoid(-2.0), sigmoid_from_module(-2.0))  # both ~0.12

# --- To verify that the sigmoid from our module is the same as the one we defined inline, 
# we can run some assertions, if the conditions are not met this will raise an error.
assert np.isclose(sigmoid(0.0), sigmoid_from_module(0.0)), "Sigmoid outputs differ at x=0.0" 
assert np.isclose(sigmoid(2.0), sigmoid_from_module(2.0)), "Sigmoid outputs differ at x=2.0"
assert np.isclose(sigmoid(-2.0), sigmoid_from_module(-2.0)), "Sigmoid outputs differ at x=-2.0" 

> **Iterating on the module.** If you edit `src/qmn_utils.py` while the notebook kernel is already running, Python keeps using the *first* version it imported — your edits will be invisible until you restart the kernel. To pick up your edits *without* a kernel restart, reload the module explicitly:
>
> ```python
> import importlib, src.qmn_utils
> importlib.reload(src.qmn_utils)
> from src.qmn_utils import sigmoid    # re-bind the name in the notebook
> ```

> **▶ Task 3.5 — Move your `my_gaussian` into the module.**
> Add the `my_gaussian(x, mu, sigma)` function you wrote in Task 3.1 to `src/qmn_utils.py` (open it in VS Code, paste your function, save). Then reload the module with the trick above, import `my_gaussian` from it in the cell below, and reproduce the bell-curve plot. From now on, anything you add to `qmn_utils.py` is reusable across every notebook of the course.


In [ ]:
# TODO 3.5 — import my_gaussian from src.qmn_utils and plot it
# (1) Add `my_gaussian(x, mu, sigma)` to src/qmn_utils.py and save the file.

# (2) Reload the module (see the importlib.reload box above).
import importlib, src.qmn_utils # TO BE REMOVED
importlib.reload(src.qmn_utils) # TO BE REMOVED

# (3) Uncomment the lines below and run the cell to check that your import works.
from src.qmn_utils import my_gaussian # TO BE COMMENTED OUT
fig, ax = plt.subplots() # TO BE COMMENTED OUT
ax.plot(x_grid, my_gaussian(x_grid, mu=0.2, sigma=0.1), lw=2, color="#3a12ec", label="mu=0.2, sigma=0.1") # TO BE COMMENTED OUT
ax.legend() # TO BE COMMENTED OUT
ax.set_xlabel("x"); ax.set_ylabel("g(x)") # TO BE COMMENTED OUT
ax.set_title("Un-normalized Gaussian (from src.qmn_utils)") # TO BE COMMENTED OUT
plt.show() # TO BE COMMENTED OUT

<a id="section-4"></a>
## 4. Fundamental functions in action

Now we put the building blocks to work and look at the data through the lens of the function families from Lecture 2.

<a id="section-4-1"></a>
### 4.1 The sigmoid: psychometric curves

For each value of motion coherence, we can compute the **fraction of "rightward" responses**: the empirical probability that the subject answered "right". Plotted against coherence, this gives the **psychometric curve**, which classically has the shape of a sigmoid:

$$
P(\text{right} \mid \text{coh}) = \sigma(\text{coh}; k, x_0).
$$

The parameter $k$ measures the subject's **sensitivity**: steeper curves correspond to higher sensitivity. The parameter $x_0$ captures any **left/right bias**.

Let's compute the psychometric curve for subject `S01`, **pooling all sessions** for now. We will use the pandas `.groupby()` method to group trials by motion coherence and compute the mean response within each group.

The basic idea of `.groupby()` is:

```python
df.groupby("some_column")
```

This tells pandas:

> split the DataFrame into separate groups, one for each unique value of `"some_column"`.

After creating these groups, we usually apply a **summary function** to each group, such as `.mean()`, `.sum()`, `.count()`, `.median()`, or `.std()`. This reduces many rows into one summary value per group.

For example:

```python
df.groupby("some_column")["another_column"].mean()
```

means:

> group the rows by `"some_column"`, then compute the mean of `"another_column"` separately within each group.

In our case:

```python
s01.groupby("coherence")["choice_right"].mean()
```

means:

1. split the trials from subject `S01` into groups with the same coherence value;
2. take the column `"choice_right"` inside each group;
3. compute the mean of `"choice_right"` for each coherence value.

If `"choice_right"` is coded as:

```text
0 = leftward choice
1 = rightward choice
```

then the mean of `"choice_right"` is exactly the fraction of rightward choices:

$$
\text{mean choice\_right} = P(\text{right} \mid \text{coh}).
$$

So `.groupby()` lets us go from trial-by-trial data to one summary value per coherence level.

In [ ]:
# --- Empirical psychometric curve for one subject
subj = "S01"
sub = df[df["subject_id"] == subj]

# Group trials by coherence and compute the fraction of "right" responses
psy = sub.groupby("coherence")["response"].mean()
print(psy.round(3))

In [ ]:
# --- Plot it, with a sigmoid model overlay
fig, ax = plt.subplots()
ax.plot(psy.index, psy.values, "o", ms=8, label="data (subject S01)")

# A hand-chosen sigmoid for comparison — we'll fit one properly in a later notebook
x = np.linspace(-0.6, 0.6, 200)
ax.plot(x, sigmoid(x, slope=8.0, bias=0.0), "-", lw=2, label="sigmoid model")

ax.axhline(0.5, color="gray", lw=0.8, ls="--")
ax.axvline(0.0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("signed motion coherence")
ax.set_ylabel("P(response = right)")
ax.set_title(f"Psychometric curve — subject {subj}")
ax.legend()
plt.show()

> **▶ Task 4.1 — Compare subjects.**
> Reuse the recipe above to plot the psychometric curves of **all three subjects on the same figure**. Tip: a `for subj in df["subject_id"].unique(): ...` loop is the cleanest way.
>
> Once your plot is done, eyeball it: which subject looks most *sensitive* (steepest curve)? Which one looks *biased* (curve not centered on 0)?


In [ ]:
# TODO 4.1 — psychometric curves for all subjects on one figure
fig, ax = plt.subplots()

for subj in df["subject_id"].unique():
    # 1) select the subject's trials
    sub = df[df["subject_id"] == subj] # TO BE REMOVED
    # 2) compute fraction of "right" responses per coherence
    curr_psy = sub.groupby("coherence")["response"].mean() # TO BE REMOVED
    # 3) ax.plot(...) with a marker and a label=subj
    ax.plot(curr_psy.index, curr_psy.values, marker="o", ls="none", label=subj) # TO BE REMOVED

ax.axhline(0.5, color="gray", lw=0.8, ls="--")
ax.set_xlabel("signed motion coherence")
ax.set_ylabel("P(response = right)")
ax.set_title("Psychometric curves — all subjects")
ax.legend()
plt.show()

<a id="section-4-2"></a>
### 4.2 Exponential and logarithm: reaction time distributions

Reaction times are *strictly positive* and typically *positively skewed*: long tail of slow trials, no symmetric short-trial counterpart. A standard trick is to look at $\log(\text{RT})$ instead of $\text{RT}$: on the log scale the distribution becomes much more symmetric and well-described by a Gaussian. This is the defining property of a **log-normal** random variable.

The logarithm also turns *multiplicative* effects into *additive* ones, and *exponential* trends into *straight lines*. This is exactly the trick illustrated in the lecture's *log-scale* slide.

Let's see it on the RTs of subject `S01`.

In [ ]:
# --- Side-by-side: histogram of RTs and histogram of log(RTs)
rts = df.loc[df["subject_id"] == "S01", "rt_s"].values

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(rts, bins=40, color="C0", edgecolor="white")
axes[0].set_xlabel("reaction time (s)")
axes[0].set_ylabel("count")
axes[0].set_title("RT — linear scale")

axes[1].hist(np.log(rts), bins=40, color="C1", edgecolor="white")
axes[1].set_xlabel("log(reaction time / s)")
axes[1].set_ylabel("count")
axes[1].set_title("log(RT) — much more symmetric")

plt.tight_layout()
plt.show()

> **▶ Task 4.2 — Log-scale plotting.**
> Matplotlib lets you change the scale of an axis with `ax.set_xscale("log")` and `ax.set_yscale("log")`. Re-plot the *linear-scale* histogram above with a **logarithmic x-axis** (no need to take the log of the data yourself: Matplotlib will). What do you observe? How does this connect with the log-RT histogram on the right?


In [ ]:
# TODO 4.2 — RT histogram with a logarithmic x-axis
fig, ax = plt.subplots()
# Hint: ax.hist(rts, bins=np.logspace(...)) gives the nicest result on a log axis.
ax.hist(rts, bins=np.logspace(np.log10(rts.min()), np.log10(rts.max()), 40), edgecolor="white", color="C5") # TO BE REMOVED
ax.set_xscale("log") # TO BE REMOVED
ax.set_xticks([0.25, 0.5, 1.0, 2.0])
ax.set_xticklabels(["0.25", "0.5", "1.0", "2.0"])
ax.set_xlabel("reaction time (s)")
ax.set_ylabel("count")
ax.set_title("RT histogram with logarithmic x-axis")   

plt.show()

<a id="section-4-3"></a>
### 4.3 Gaussian: the bell-shaped friend

Now that the log-RTs look symmetric, let's overlay a **Gaussian** with the empirical mean and standard deviation. This is the standard way to assess whether a log-normal model is a sensible description of RTs.

Recall the (properly normalized) Gaussian probability density:

$$ \mathcal{N}(x;\, \mu, \sigma) \;=\; \frac{1}{\sigma\sqrt{2\pi}}\;\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right). $$


In [ ]:
# --- Gaussian overlay on log-RT histogram
log_rts = np.log(rts)
mu_hat = log_rts.mean()
sg_hat = log_rts.std()

x = np.linspace(log_rts.min(), log_rts.max(), 200)
pdf = (1.0 / (sg_hat * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu_hat) / sg_hat) ** 2)

fig, ax = plt.subplots()
ax.hist(log_rts, bins=40, density=True, color="C1", edgecolor="white", alpha=0.7,
        label="empirical")
ax.plot(x, pdf, "k-", lw=2, label=fr"$\mathcal{{N}}(\mu={mu_hat:.2f},\ \sigma={sg_hat:.2f})$")
ax.set_xlabel("log(reaction time / s)")
ax.set_ylabel("density")
ax.set_title("log-RTs look approximately Gaussian")
ax.legend()
plt.show()

<a id="section-4-4"></a>
### 4.4 Sine and cosine: a quick LFP-flavoured detour

We don't have an LFP recording in our dataset, but sine and cosine functions deserve a short illustration because they are the basic units of every oscillation you will meet in EEG, MEG and LFP analyses.

Here is a quick synthetic example: a 10 Hz oscillation (alpha-band-ish) plus a bit of noise.

In [ ]:
# --- A synthetic 10 Hz "alpha" oscillation
fs = 1000                              # sampling rate (Hz)
t = np.arange(0, 1.0, 1 / fs)          # one second of data
freq = 10                              # Hz
signal = np.sin(2 * np.pi * freq * t) + 0.3 * np.random.default_rng(0).standard_normal(t.size)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, signal, lw=1)
ax.set_xlabel("time (s)")
ax.set_ylabel("amplitude (a.u.)")
ax.set_title("Toy 10 Hz oscillation + noise")
plt.show()

> **▶ Task 4.3 — Build an LFP from a *sum* of oscillations.**
> Real neural signals are mixtures of multiple frequencies. Build a new signal that is the sum of three sine waves:
> - 4 Hz (theta) with amplitude 0.5
> - 10 Hz (alpha) with amplitude 0.2
> - 40 Hz (gamma) with amplitude 0.3
>
> Plot the resulting signal. (We will see in Week 13 / Fourier analysis how to *recover* the three components from the mixture.)


In [ ]:
# TODO 4.3 — sum of three oscillations
fs = 1000
t = np.arange(0, 1.0, 1 / fs)

# Replace the line below with the sum of three sinusoids (4, 10 and 40 Hz)
sin_1 = 0.5 * np.sin(2 * np.pi * 4 * t) # TO BE REMOVED
sin_2 = 0.2 * np.sin(2 * np.pi * 10 * t) # TO BE REMOVED
sin_3 = 0.3 * np.sin(2 * np.pi * 40 * t) # TO BE REMOVED
noise = 0.3 * np.random.default_rng(0).standard_normal(t.size) # to make it look more like real data

signal = (sin_1 + 
          sin_2 + 
          sin_3 + 
          noise) # TO BE REMOVED

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, signal, lw=1)
ax.set_xlabel("time (s)")
ax.set_ylabel("amplitude (a.u.)")
ax.set_title("Sum of three oscillations (4, 10, 40 Hz) + noise")
plt.show()

<a id="section-5"></a>
## 5. Learning curves and a bit of calculus

Across the 8 sessions, our subjects get better at the task. We expect their **accuracy** to grow and their **RTs** to shrink — and both trajectories are well-approximated by an **exponential approach to an asymptote**:

$$ y(t) \;=\; y_\infty \;+\; (y_0 - y_\infty)\,e^{-t/\tau} $$

where $y_0$ is the starting value, $y_\infty$ the asymptotic value, and $\tau$ the *learning time-constant* (in sessions).

<a id="section-5-1"></a>
### 5.1 Accuracy across sessions: an exponential learning curve

In [ ]:
# --- Mean accuracy per session, per subject
learning = (
    df.dropna(subset=["correct"]) # drop trials (i.e. rows) with missing "correct" values
      .groupby(["subject_id", "session"])["correct"] # group by subject and session, and select the "correct" column
      .mean() # compute mean accuracy per subject and session
      .reset_index() # convert to a regular DataFrame with columns "subject_id", "session", "correct"
)
learning.tail(10)

In [ ]:
# --- Plot one learning curve per subject
fig, ax = plt.subplots()
for subj, sub in learning.groupby("subject_id"):
    ax.plot(sub["session"], sub["correct"], "o-", label=subj)
ax.set_xlabel("session")
ax.set_ylabel("mean accuracy")
ax.set_ylim(0.5, 1.0)
ax.set_title("Learning curves across 8 sessions - all subjects")
ax.legend()
plt.show()

Let's also overlay the *exponential-approach* template we wrote down above. We won't fit it (we will see proper curve fitting in Week 7 — Regression). Instead, we pick a few reasonable values by eye and look at the shape.

In [ ]:
# --- Hand-picked exponential approach to compare with the data
def exp_approach(t, y0, y_inf, tau):
    """y_inf + (y0 - y_inf) * exp(-t/tau)"""
    return y_inf + (y0 - y_inf) * np.exp(-t / tau)

t_grid = np.linspace(0, 7, 200) # sessions are numbered 1..8, so t = session - 1
model = exp_approach(t_grid, y0=0.65, y_inf=0.88, tau=2.5)

fig, ax = plt.subplots()
for subj, sub in learning.groupby("subject_id"):
    ax.plot(sub["session"], sub["correct"], "o", label=subj)
ax.plot(t_grid + 1, model, "k-", lw=2, label=r"$y_\infty + (y_0 - y_\infty)\,e^{-t/\tau}$")
ax.set_xlabel("session")
ax.set_ylabel("mean accuracy")
ax.set_ylim(0.5, 1.0)
ax.set_title("Learning curve with an exponential-approach overlay")
ax.legend()
plt.show()

> **▶ Task 5.1 — RT learning curve.**
> Repeat the analysis above for the **mean reaction time** of each subject across sessions, and plot the three curves. Does the time-constant of the decay look similar to the accuracy one, or different?


In [ ]:
# TODO 5.1 — RT learning curve
rt_learning = (
    df.groupby(["subject_id", "session"])["rt_s"] # group by subject and session, and select the "rt_s" column
      .mean() # then compute mean RT per subject and session
      .reset_index() # convert to a regular DataFrame with columns "subject_id", "session", "rt_s"
)

fig, ax = plt.subplots()
# TODO: plot one curve per subject, with appropriate labels
for subj, sub in rt_learning.groupby("subject_id"): # TO BE REMOVED
    ax.plot(sub["session"], sub["rt_s"], "o-", label=subj) # TO BE REMOVED  
ax.set_xlabel("session")
ax.set_ylabel("mean reaction time (s)")
ax.legend()
plt.show()

<a id="section-5-2"></a>
### 5.2 Numerical derivative: rate of learning

The **derivative** of a function tells us how fast it is changing. Even though we don't have an explicit formula for the learning curve, we can estimate the derivative *numerically* from the discrete points: a simple finite difference

$$ \frac{dy}{dt}\Big|_{t_i} \;\approx\; \frac{y_{i+1} - y_{i-1}}{t_{i+1} - t_{i-1}} $$

is exactly what `np.gradient` implements for us (https://numpy.org/devdocs/reference/generated/numpy.gradient.html).

In [ ]:
# --- Numerical derivative of the learning curve, per subject
fig, ax = plt.subplots()
for subj, sub in learning.groupby("subject_id"): # loop over subjects ids obtained from the "subject_id" column of the "learning" DataFrame
    sessions = sub["session"].values
    acc      = sub["correct"].values
    d_acc    = np.gradient(acc, sessions) # finite-difference derivative
    ax.plot(sessions, d_acc, "o-", label=subj)

ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("session")
ax.set_ylabel(r"$\Delta$ accuracy per session")
ax.set_title("Rate of learning (numerical derivative of the learning curve)")
ax.legend()
plt.show()

**Interpretation.** Positive values mean the subject is still improving; values close to zero mean they have *plateaued*. Because the learning curve is concave (fast at the start, slow later), the derivative is largest in the first few sessions and decays toward zero — this is precisely the *exponential decay* of the derivative of $e^{-t/\tau}$ (lecture: derivative of the exponential).

<a id="section-5-3"></a>
### 5.3 Numerical integral: cumulative correct responses

Where the derivative captures *rate of change*, the **integral** captures *accumulation*. For discrete observations the natural counterpart of the integral is the **cumulative sum**. Numpy function `np.cumsum` will implement it for us (https://numpy.org/devdocs/reference/generated/numpy.cumsum.html).

A neuroscience-friendly example: how many correct responses has a subject accumulated up to trial $n$? This is just the running total of the `correct` column.

In [ ]:
# --- Cumulative correct responses, one curve per subject
fig, ax = plt.subplots()
for subj, sub in df.groupby("subject_id"):
    sub = sub.sort_values(["session", "trial_in_session"]).copy() # sort trials in chronological order within each subject, and make a copy to avoid writing on the original DataFrame
    sub["cum_correct"] = sub["correct"].fillna(0).cumsum() # create a new column with the cumulative sum of correct responses, treating missing values as 0
    ax.plot(sub["trial_id"] - sub["trial_id"].min(), sub["cum_correct"], lw=1.2, label=subj) # plot cumulative correct responses against trial number (starting from 0 for each subject)

ax.set_xlabel("trial number (within subject)")
ax.set_ylabel("cumulative correct responses")
ax.set_title("Cumulative correct responses — discrete integral of `correct`")
ax.legend()
plt.show()

**Interpretation.** The *slope* of these cumulative curves at trial $n$ is the local *accuracy* at that trial — by the Fundamental Theorem of Calculus, derivative and cumulative sum undo each other. The fact that the curves *bend upward* means accuracy is increasing across trials, exactly what the learning curve told us in Section 5.1.

<a id="section-6"></a>
## 6. Exercises

> **▶ Task 6.1 — Difficulty-dependent RT.**
> For subject `S01`, compute the **mean RT per coherence magnitude** (use `np.abs(coherence)`) and plot mean RT against $|\text{coherence}|$. Does the curve look like one of the functions from Lecture 2? If yes, which one?
> 
> *Hint.* Use again `.groupby()` and `.mean()` with `.index` and `.values` to select what you need.



In [ ]:
# TODO 6.1 — mean RT vs |coherence| (with distribution of RTs across coherence levels in the background)
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=True)
subject_axes = {
    "S01": axes[0],
    "S02": axes[1],
    "S03": axes[2],
}
subject_colors = {
    "S01": "C0",
    "S02": "C1",
    "S03": "C2",
}
for subj, sub in df.groupby("subject_id"):
    ax = subject_axes[subj]
    coh = sub["coherence"].abs()
    rt = sub["rt_s"]
    ax.plot( # TO BE REMOVED
        coh, # TO BE REMOVED
        rt, # TO BE REMOVED
        "o", # TO BE REMOVED
        label=subj, # TO BE REMOVED
        markersize=3, # TO BE REMOVED
        alpha=0.3, # TO BE REMOVED
        color=subject_colors[subj], # TO BE REMOVED
    ) # TO BE REMOVED
    ax.plot( # TO BE REMOVED
        coh.groupby(coh).mean().index, # TO BE REMOVED
        rt.groupby(coh).mean().values, # TO BE REMOVED
        "-", # TO BE REMOVED
        color="black", # TO BE REMOVED
        label=f"{subj} mean", # TO BE REMOVED
    ) # TO BE REMOVED
    ax.set_title(subj)
    ax.legend()
    ax.set_xlabel("|coherence|")
fig.suptitle("Mean RT vs |coherence|", y=1.03)
fig.supylabel("reaction time (s)")
plt.tight_layout()
plt.show()

> **▶ Task 6.2 — A sigmoid per session.**
> Pick one subject. Compute its empirical psychometric curve **separately for each session**, and plot all 8 curves on the same axes with a sequential colour scheme (e.g. `cmap = plt.cm.viridis`). Does the curve get *steeper* across sessions? Relate your observation to the *slope* of the learning curve in Section 5.1. If you feel inspired try to plot the curves of all three subjects in the same subplots.

In [ ]:
# TODO 6.2 — one psychometric curve per session
fig, axs = plt.subplots(2,4, figsize=(16, 8), sharex=True, sharey=True)
# subj = "S02" # TO BE REMOVED
for subj in df["subject_id"].unique(): # TO BE REMOVED
    sub = df[df["subject_id"] == subj] # TO BE REMOVED
    color = {"S01": "C0", "S02": "C1", "S03": "C2"}[subj] # TO BE REMOVED
    axs = axs.flatten()
    for i, (session, session_data) in enumerate(sub.groupby("session")):
        psy = session_data.groupby("coherence")["response"].mean() # TO BE REMOVED
        axs[i].plot(psy.index, psy.values, "o-", color=color, label=f"session {session}") # TO BE REMOVED
        axs[i].axhline(0.5, color="gray", lw=0.8, ls="--")
        axs[i].axvline(0.0, color="gray", lw=0.8, ls="--")
        axs[i].set_xlabel("signed motion coherence")
        axs[i].set_ylabel("P(response = right)")
        axs[i].legend(loc="lower right")
# fig.suptitle(f"Psychometric curves across sessions — subject {subj}")
fig.suptitle(f"Psychometric curves across sessions — all subjects")
plt.show()

> **▶ Task 6.3 — Spot the bias.**
> Look at the data of subject `S01` again. Their `coherence = 0` trials carry **no objective correct answer**, but they reveal any *response bias*: if the subject answered "right" more than 50% of the time at coherence 0, they are biased toward "right". Compute this proportion and compare with the other two subjects.


In [ ]:
# TODO 6.3 — response bias at coherence 0
fig, ax = plt.subplots()
for subj, sub in df.groupby("subject_id"):
    zero_coh = sub.loc[sub["coherence"] == 0, "response"] # TO BE REMOVED
    p_right = zero_coh.mean() # TO BE REMOVED
    n = zero_coh.count() # number of trials with zero coherence
    se = np.sqrt(p_right * (1 - p_right) / n) # correct uncertainty of a proportion with binomial standard error (see next lectures for details)
    ax.bar(
        subj,
        p_right,
        color=subject_colors[subj],
    )
    ax.errorbar(
        subj,
        p_right,
        yerr=se,
        color="black",
        capsize=5,
        linestyle="none",
    )
ax.axhline(0.5, color="gray", lw=0.8, ls="--")
ax.set_ylabel("P(response = right) at coherence 0")
ax.set_title("Response bias at coherence 0")

plt.show()

> **▶ Task 6.4 — Your own neuroscience function.**
> Many neural signals follow an **exponentially decaying response** — for instance the calcium transient triggered by a single action potential, or an EPSP after a presynaptic spike. Write a function `ca_transient(t, amplitude, tau)` that returns
> $$ f(t) \;=\; A\,e^{-t/\tau} \quad\text{for } t \ge 0,\qquad 0 \text{ otherwise.} $$
> Plot it for $A = 1$, $\tau = 0.2\,\text{s}$, on a 1-second time grid. Then plot it again on a **semi-log y-axis** (`ax.set_yscale("log")`) and verify that the curve becomes a straight line — the log-scale trick from the lecture.
>
> Then compute integral and derivative of the signals.
> 
> *Hint:* use `np.where(...)` to define the function differently before and after `t = 0`.
>
> ```python
> np.where(condition, value_if_true, value_if_false)
> ```
>
> In this case, the condition should check whether `t >= 0`.


In [ ]:
# TODO 6.4 — calcium transient

# Visualize the trace on a linear and a semi-log y-axis
def ca_transient(t, amplitude, tau):
    # TODO: return amplitude * exp(-t / tau) for t >= 0, and 0 for t < 0
    trace = np.where(t >= 0, amplitude * np.exp(-t / tau), 0) # TO BE REMOVED
    return trace

t = np.linspace(-1, 1.0, 1000)
trace = ca_transient(t, amplitude=1.0, tau=0.2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t, trace, lw=2, color="black")
axes[0].set_xlabel("time (s)")
axes[0].set_ylabel("calcium transient (a.u.)")
axes[0].set_title("Calcium transient — linear y-axis")      
# On a log y-axis, the t < 0 part of the trace is zero (i.e. -inf in log) — restrict to t >= 0
mask = t >= 0
axes[1].plot(t[mask], trace[mask], lw=2, color="black")
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("calcium transient (a.u.)")
axes[1].set_title("Calcium transient — semi-log y-axis (t >= 0 only)")
axes[1].set_yscale("log")
plt.tight_layout()
plt.show()

# Visualize their integral (cumulative sum) and derivative (finite-difference) as well
cumulative_trace = np.cumsum(trace) # TO BE REMOVED
derivative_trace = np.gradient(trace, t) # TO BE REMOVED

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t, cumulative_trace, lw=2, color="C0")
axes[0].set_xlabel("time (s)")
axes[0].set_ylabel("cumulative sum (a.u. s)")
axes[0].set_title("Cumulative sum (integral) of the transient")
axes[1].plot(t, derivative_trace, lw=2, color="C0")
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("finite-difference derivative (a.u. / s)")
axes[1].set_title("Finite-difference derivative of the transient")
plt.tight_layout()